# Module 19 — SQL with SQLite

The same fifty readings as module 18, in a database — plus a second table saying
where each sensor is. `sqlite3` is in the standard library, so there is nothing to
install and no server to run: a SQLite database is one file.

Two things this module is for.

**Never build a query by joining strings.** Section 3 shows the same value going
through a placeholder and through an f-string: one finds nothing, the other returns
every row in the table. That is not a style preference.

**And a `REAL` column will accept the string `"kaputt"`.** Section 7 measures it.
Module 18 was the fourth time this course met a tool that guesses instead of
refusing; this is the fifth, and it is in the place where you would most expect a
type to be enforced.

Run `uv run 19_sql/build_db.py` once before you start — it builds `data/readings.db`
from the two CSV files, and the file is gitignored because a database is a build
artefact.

In [ ]:
import sqlite3
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "build_db.py").is_file() else Path.cwd() / "19_sql"
DB = HERE / "data" / "readings.db"

print(sqlite3.sqlite_version)
print(DB.exists(), DB.stat().st_size, "bytes")

## 1. Connect, query, close

`sqlite3.connect` on a path opens or creates the file. The connection is a context
manager, but **not** the one you expect: `with connection:` is a *transaction*, not a
close. Section 8 comes back to that; for reading, `contextlib.closing` is the honest
form.

In [ ]:
import contextlib

with contextlib.closing(sqlite3.connect(DB)) as connection:
    rows = connection.execute("SELECT tag, location FROM sensors ORDER BY tag").fetchall()

print(type(rows[0]).__name__)
for row in rows:
    print(row)

Rows arrive as **tuples**, positionally. That is fine for two columns and awful for
seven, so set `row_factory` and get something you can index by name:

In [ ]:
with contextlib.closing(sqlite3.connect(DB)) as connection:
    connection.row_factory = sqlite3.Row  # do this before you execute anything
    row = connection.execute("SELECT tag, location, installed FROM sensors LIMIT 1").fetchone()

print(row["tag"], row["location"])
print(row.keys())
print(dict(row))  # straight into json.dumps, module 08

`fetchone()` gives one row or `None`; `fetchall()` gives a list. And the cursor is
**iterable**, which is module 13 arriving where it matters: iterating fetches in
batches rather than building the whole list, so a query returning ten million rows
does not have to fit in memory.

In [ ]:
with contextlib.closing(sqlite3.connect(DB)) as connection:
    cursor = connection.execute("SELECT tag, value FROM readings WHERE value > 85")
    for tag, value in cursor:  # tuple unpacking in the loop header -- module 05
        print(tag, value)

## 2. SELECT

The clauses, in the order SQL requires them written:

```sql
SELECT   tag, value          -- which columns
FROM     readings            -- from where
WHERE    value > 85          -- which rows
GROUP BY tag                 -- collapsed how
HAVING   COUNT(*) > 1        -- which groups
ORDER BY value DESC          -- in what order
LIMIT    5                   -- how many
```

The order they *execute* in is different — `FROM`, `WHERE`, `GROUP BY`, `HAVING`,
`SELECT`, `ORDER BY`, `LIMIT` — which is why `WHERE` cannot use a column alias
defined in `SELECT` and `ORDER BY` can.

In [ ]:
with contextlib.closing(sqlite3.connect(DB)) as connection:
    connection.row_factory = sqlite3.Row
    for row in connection.execute(
        """
        SELECT tag, value, at
        FROM   readings
        WHERE  value > 85
        ORDER  BY value DESC
        LIMIT  3
        """
    ):
        print(row["tag"], row["value"], row["at"])

And the difference from `WHERE` on a `HAVING`: `WHERE` filters rows before grouping,
`HAVING` filters groups after. Getting them the wrong way round is the most common
SQL mistake after the one in section 3.

## 3. Placeholders, and why this one is not a style question

You need a query parameterised by a tag. There are two ways to write it, and one of
them is a security bug.

In [ ]:
tag = "TH-04"

with contextlib.closing(sqlite3.connect(DB)) as connection:
    safe = connection.execute(
        "SELECT COUNT(*) FROM readings WHERE tag = ?",
        (tag,),  # a tuple, always
    ).fetchone()[0]

print(safe, "readings for", tag)

`?` is a **placeholder**. The value is sent to SQLite separately from the query text,
so it is never parsed as SQL — it cannot be anything but a value.

Now the same query with the value pasted in, and a value chosen to show why that
matters. Predict both numbers.

In [ ]:
hostile = "TH-04' OR '1'='1"

with contextlib.closing(sqlite3.connect(DB)) as connection:
    with_placeholder = connection.execute(
        "SELECT COUNT(*) FROM readings WHERE tag = ?", (hostile,)
    ).fetchone()[0]

    with_fstring = connection.execute(
        f"SELECT COUNT(*) FROM readings WHERE tag = '{hostile}'"  # noqa: S608 -- the point
    ).fetchone()[0]

total = 50

assert with_placeholder == ...
assert with_fstring == ...

The placeholder found **nothing**: there is no sensor with that tag, which is the
truthful answer. The f-string returned **all fifty rows**, because what SQLite parsed
was

```sql
SELECT COUNT(*) FROM readings WHERE tag = 'TH-04' OR '1'='1'
```

and `'1'='1'` is true for every row. The `WHERE` clause was rewritten by the data.

This is **SQL injection**, and the version above is the mild one — it reads too much.
Measured, with the same hole in different queries:

- On a `DELETE FROM readings WHERE tag = '...'`, the value `x' OR '1'='1` deletes
  **all fifty rows**. One statement, whole table.
- On a `SELECT`, the value `x' UNION SELECT tag, location, installed FROM sensors --`
  returns rows from a table the query never mentioned.
- On a login form, `' OR '1'='1` in the password field matches the first user.

The textbook payload `'; DROP TABLE readings; --` is worth a note, because it does
**not** work here: `execute` refuses more than one statement —
`ProgrammingError: You can only execute one statement at a time`. It works through
`executescript`, which is the reason never to hand that foreign text. But the three
above need only one statement, so the protection is thin comfort.

The rule, and it has no exceptions worth learning: **values go in placeholders.**

```python
connection.execute("... WHERE tag = ?", (tag,))                   # yes
connection.execute("... WHERE tag = :tag", {"tag": tag})          # yes, named
connection.execute(f"... WHERE tag = '{tag}'")                    # no
connection.execute("... WHERE tag = '" + tag + "'")               # no
```

Two things people get wrong about the rule:

- **The parameter must be a sequence.** `(tag,)` with the comma — module 05's
  one-element tuple. `(tag)` is just `tag`, and you get `ValueError`.
- **A placeholder cannot stand in for a table or column name.** `SELECT ? FROM
  readings` selects the *string* you passed, once per row. Where the column name is
  genuinely dynamic, check it against a list of allowed names yourself; there is no
  placeholder that will do it for you.

In [ ]:
with contextlib.closing(sqlite3.connect(DB)) as connection:
    # A placeholder in a column position: SQLite treats it as a value.
    print(connection.execute("SELECT ? FROM readings LIMIT 2", ("value",)).fetchall())

    # The named form, which reads better once there are three of them.
    print(
        connection.execute(
            "SELECT COUNT(*) FROM readings WHERE tag = :tag AND value > :limit",
            {"tag": "TH-04", "limit": 85},
        ).fetchone()[0]
    )

## 4. Two tables, and the reason there are two

`readings` says a tag; `sensors` says where that tag is. Storing the location on every
reading instead would repeat it fifty times, and then a sensor that moves has to be
corrected in fifty places — or, more likely, in forty-nine.

That is **normalisation**: each fact in one place. `JOIN` is how the two come back
together.

In [ ]:
with contextlib.closing(sqlite3.connect(DB)) as connection:
    connection.row_factory = sqlite3.Row
    for row in connection.execute(
        """
        SELECT   s.location, COUNT(r.value) AS n, ROUND(AVG(r.value), 2) AS mean
        FROM     readings r
        JOIN     sensors  s ON s.tag = r.tag
        GROUP BY s.location
        ORDER BY mean DESC
        """
    ):
        print(f"{row['location']:<10}{row['n']:>3}{row['mean']:>8}")

Compare that with module 18's `groupby("location")` — the same three numbers, and the
same shape of answer. The difference is where the work happens: pandas read fifty
rows into memory and grouped them there, SQLite grouped them in the file and returned
three rows.

`r` and `s` are **aliases**, and `s.tag = r.tag` is the join condition. Leave the
condition out and you get every combination of every row — a cross join, 250 rows
here and a very long wait on a real table.

### The rows a JOIN drops

A plain `JOIN` keeps only rows that match on **both** sides. So a reading whose tag is
not in `sensors` disappears — silently, from a query that ran successfully.

In [ ]:
with contextlib.closing(sqlite3.connect(DB)) as connection:
    connection.execute(
        "INSERT INTO readings (tag, value, at) VALUES (?, ?, ?)",
        ("TH-77", 42.0, "2026-03-01T18:00"),  # a tag with no row in sensors
    )

    joined = connection.execute(
        "SELECT COUNT(*) FROM readings r JOIN sensors s ON s.tag = r.tag"
    ).fetchone()[0]
    all_rows = connection.execute("SELECT COUNT(*) FROM readings").fetchone()[0]

    print(all_rows, "readings,", joined, "survive the join")

    for row in connection.execute(
        """
        SELECT   r.tag
        FROM     readings r
        LEFT JOIN sensors s ON s.tag = r.tag
        WHERE    s.tag IS NULL
        """
    ):
        print("orphan:", row[0])

    connection.rollback()  # undo the insert; section 8 is about this line

`LEFT JOIN` keeps every row on the left and fills the right with `NULL` where there is
no match — which is how you *find* the orphans instead of losing them. `WHERE s.tag IS
NULL` after a `LEFT JOIN` is the standard idiom for "rows on the left with nothing on
the right", and it is worth recognising on sight.

The schema in `build_db.py` declares `REFERENCES sensors(tag)`, which says this should
be impossible. **SQLite does not enforce that unless you ask it to** — foreign keys
are off by default, for backwards compatibility:

In [ ]:
with contextlib.closing(sqlite3.connect(DB)) as connection:
    print("foreign_keys:", connection.execute("PRAGMA foreign_keys").fetchone()[0])

    connection.execute("PRAGMA foreign_keys = ON")
    print("after asking: ", connection.execute("PRAGMA foreign_keys").fetchone()[0])

    try:
        connection.execute(
            "INSERT INTO readings (tag, value, at) VALUES (?, ?, ?)",
            ("TH-77", 42.0, "2026-03-01T18:00"),
        )
    except sqlite3.IntegrityError as err:
        print("IntegrityError:", err)

    connection.rollback()

So `PRAGMA foreign_keys = ON` after every connect, if you want the constraint you
wrote down to mean anything. It is one line, it has to be repeated per connection, and
forgetting it is how a database ends up with orphans that the schema says cannot
exist.

## 5. Grouping, and what COUNT counts

`COUNT(*)` counts **rows**. `COUNT(column)` counts rows where that column is **not
NULL**. The three unreadable readings are `NULL` in the database, so the two numbers
differ — which is module 18's `len()` against `count()`, in SQL.

In [ ]:
with contextlib.closing(sqlite3.connect(DB)) as connection:
    rows = connection.execute("SELECT COUNT(*), COUNT(value) FROM readings").fetchone()

# 50 readings, 3 of them NULL. What are the two numbers?
assert rows == ...

In [ ]:
with contextlib.closing(sqlite3.connect(DB)) as connection:
    connection.row_factory = sqlite3.Row
    for row in connection.execute(
        """
        SELECT   s.location,
                 COUNT(*)        AS readings,
                 COUNT(r.value)  AS usable,
                 ROUND(AVG(r.value), 2) AS mean,
                 MAX(r.value)    AS highest
        FROM     readings r
        JOIN     sensors  s ON s.tag = r.tag
        GROUP BY s.location
        ORDER BY s.location
        """
    ):
        print(
            f"{row['location']:<10}{row['readings']:>3}{row['usable']:>4}"
            f"{row['mean']:>8}{row['highest']:>7}"
        )

That is exercise 09 of module 18, in one query. `AVG` ignores `NULL` exactly as
pandas' `mean()` does, so a summary with only `COUNT(*)` in it claims readings it does
not have — the same error, in a different language, with the same fix: report both.

And `NULL` behaves like `NaN` in the one way that catches everybody:

In [ ]:
with contextlib.closing(sqlite3.connect(DB)) as connection:
    print(connection.execute("SELECT NULL = NULL, NULL IS NULL").fetchone())
    print(
        connection.execute("SELECT COUNT(*) FROM readings WHERE value IS NULL").fetchone()[0],
        "found with IS NULL",
    )
    print(
        connection.execute(
            "SELECT COUNT(*) FROM readings WHERE value = NULL"  # noqa: S608
        ).fetchone()[0],
        "found with = NULL",
    )

`NULL = NULL` is not true — it is `NULL`, which is not true either. So `WHERE value =
NULL` matches nothing, ever, and finds none of the three rows that are actually NULL.
`IS NULL` is the operator. Module 18 said the same thing about `NaN == NaN`, and it is
the same reason: a missing value is not a value that can be equal to anything.

## 6. Writing

`INSERT`, `UPDATE`, `DELETE` — with placeholders, and `executemany` for a batch.

In [ ]:
with contextlib.closing(sqlite3.connect(DB)) as connection:
    connection.executemany(
        "INSERT INTO readings (tag, value, at) VALUES (?, ?, ?)",
        [
            ("TH-01", 20.1, "2026-03-02T08:00"),
            ("TH-04", 24.4, "2026-03-02T08:00"),
        ],
    )
    print(connection.total_changes, "rows written")

    cursor = connection.execute(
        "UPDATE readings SET value = ? WHERE at LIKE ?", (0.0, "2026-03-02%")
    )
    print(cursor.rowcount, "rows updated")  # rowcount, not fetchone

    cursor = connection.execute("DELETE FROM readings WHERE at LIKE ?", ("2026-03-02%",))
    print(cursor.rowcount, "rows deleted")

    connection.rollback()
    print(connection.execute("SELECT COUNT(*) FROM readings").fetchone()[0], "rows again")

`cursor.rowcount` after an `UPDATE` or `DELETE` is how many rows it touched — and a
`DELETE` with a `WHERE` that matches nothing reports `0` rather than failing, which is
worth checking when you expected it to match something.

**A `DELETE` or `UPDATE` without a `WHERE` clause affects every row.** SQLite will not
ask. That is the one to be frightened of, and it is why the next section exists.

## 7. The types are recommendations

`value` is declared `REAL`. Predict what happens when a string goes into it.

In [ ]:
with contextlib.closing(sqlite3.connect(DB)) as connection:
    connection.execute(
        "INSERT INTO readings (tag, value, at) VALUES (?, ?, ?)",
        ("TH-01", "kaputt", "2026-03-03T08:00"),
    )
    stored = connection.execute(
        "SELECT value, typeof(value) FROM readings WHERE at = ?", ("2026-03-03T08:00",)
    ).fetchone()
    connection.rollback()

# A string, into a column declared REAL. What is in the row?
assert stored == ...

It went in, and `typeof` says `text`. SQLite has **dynamic typing**: a column's
declared type is an *affinity*, a preference for how to convert values it can convert,
not a constraint. `"kaputt"` cannot be converted to a number, so it is stored as text —
in a `REAL` column, in a row that every later `AVG` and `>` has to cope with.

Which is the fifth time this course has met the same shape:

| module | the tool | what it does instead of failing |
| --- | --- | --- |
| 08 | `latin-1` | decodes any bytes; gives you `Â°C` |
| 16 | `requests` with no charset | falls back to Latin-1; gives you `Â°C` |
| 17 | `html.parser` | repairs; gives you four cells where there are two |
| 18 | `read_csv` | picks a type from the data; `.sum()` concatenates |
| 19 | SQLite | stores what it was given; a `REAL` column holds `'kaputt'` |

And this one is the most surprising, because a database is the place you would expect
a type to be a rule. Most other databases do refuse — this is a SQLite decision, and
`STRICT` tables are its own remedy:

In [ ]:
with contextlib.closing(sqlite3.connect(DB)) as connection:
    # DROP first: CREATE TABLE is not undone by the rollback below -- sqlite3
    # commits it as it goes -- so without this line the cell works once and
    # raises "table already exists" the second time.
    connection.execute("DROP TABLE IF EXISTS strict_demo")
    connection.execute("CREATE TABLE strict_demo (value REAL) STRICT")

    try:
        connection.execute("INSERT INTO strict_demo (value) VALUES (?)", ("kaputt",))
    except sqlite3.IntegrityError as err:
        print("IntegrityError:", err)

    connection.execute("DROP TABLE strict_demo")  # and clean up after ourselves
    connection.commit()

print("STRICT needs SQLite 3.37 or later; this is", sqlite3.sqlite_version)

So: **`STRICT` on every table you create**, unless you have a reason not to. It turns
the affinity into a constraint and this whole section into a non-problem. It is one
word.

## 8. Transactions, and the `with` that is not a close

This trips up everybody who has used a context manager before, which by now is you.

```python
with sqlite3.connect(path) as connection:      # NOT a close
    ...
```

`with` on a **connection** is a transaction: it commits at the end of the block, or
rolls back if the block raised. It does **not** close the connection. So the pattern
above leaks a connection every time it runs, and the pattern for closing is
`contextlib.closing`, which is what every cell in this notebook uses.

In [ ]:
with contextlib.closing(sqlite3.connect(DB)) as connection:
    before = connection.execute("SELECT COUNT(*) FROM readings").fetchone()[0]

    try:
        with connection:  # a transaction
            connection.execute(
                "INSERT INTO readings (tag, value, at) VALUES (?, ?, ?)",
                ("TH-01", 1.0, "2026-03-04T08:00"),
            )
            raise RuntimeError("something went wrong after the insert")
    except RuntimeError as err:
        print("caught:", err)

    after = connection.execute("SELECT COUNT(*) FROM readings").fetchone()[0]
    print(before, after, "-- the insert was rolled back")

That is the property worth having: **either all of it happened or none of it did.** A
script that inserts a thousand rows and dies at row six hundred leaves the database as
it was, rather than two-thirds updated.

The combination to remember:

```python
with contextlib.closing(sqlite3.connect(path)) as connection:   # closes
    connection.execute("PRAGMA foreign_keys = ON")
    with connection:                                            # commits or rolls back
        connection.execute("INSERT ...", (value,))
```

## 9. SQL or pandas

You have now written the same summary twice. The honest answer to which is better is
that they answer different questions.

| | SQL | pandas |
| --- | --- | --- |
| where the data lives | in a file, on disk, larger than memory | in memory |
| where the work happens | in the database | in your process |
| how much comes back | the answer — three rows | as much as you selected |
| several processes at once | yes, with locking | no |
| ad-hoc exploration | clumsy: every question is a round trip and a new query | its whole point |
| joining two tables | one clause | `pd.merge`, and you must think about it |
| a plot | no | one line |
| reproducible six months later | the query is the record | see module 18, section 8 |

The pattern that follows, and it is what modules 18 and 19 have been building towards:
**let SQL reduce, and let pandas explore.** Select the rows and columns you need with a
query — which may be three rows out of ten million — and hand those to pandas. Reading
ten million rows into a DataFrame to keep three is the mistake this table exists to
prevent.

`pd.read_sql_query` does exactly that handover:

In [ ]:
import pandas as pd

with contextlib.closing(sqlite3.connect(DB)) as connection:
    frame = pd.read_sql_query(
        """
        SELECT   s.location, r.value
        FROM     readings r
        JOIN     sensors  s ON s.tag = r.tag
        WHERE    r.value IS NOT NULL
        """,
        connection,
    )

print(frame.shape, frame["value"].dtype)
print(frame.groupby("location")["value"].mean().round(2).to_dict())

Note the `WHERE r.value IS NOT NULL`: the reduction happened in SQL, so the DataFrame
has 47 rows rather than 50 and no NaN to explain. That is the division of labour in one
line.

---

`exercises/` is next: `exercise_01.py` to `exercise_06.py`, `exercise_09.py`, and two
in `thinking.md`. Run `uv run 19_sql/build_db.py` first if you have not.

That is the end of Part 4. Part 5 puts these readings in front of a person, five times
over: a web page, a data app, an API, a desktop window and a terminal interface — the
same task, so the frameworks can be compared rather than described.